In [2]:
import io
import zipfile
import requests
import frontmatter

In [3]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [4]:
repository_data = []
# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [5]:
print(repository_data[1])

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

In [6]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organisation
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download respository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md')
            or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [7]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1216
Evidently documents: 95


In [8]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [9]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [10]:
print(evidently_chunks[0].keys())

dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [11]:
import re
text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [12]:
print(paragraphs)

['In this tutorial, you will learn how to perform regression testing for LLM outputs.', 'You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.', "<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>", '# Tutorial scope', "Here's what we'll do:", '* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.', '* **Get new answers**. Imitate generating new answers to the same question.', '* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.', '* **Build a monitoring Dashboard**. Get plots to track the result

In [13]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with '## '
    header_pattern = r'^(#{' +str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    #Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is '## ', group2 is the header text
        header = parts[i] + parts[i+1] # "##" + "Title"
        header = header.strip()

        #Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections

In [14]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

In [15]:
print(evidently_chunks[0].keys())

dict_keys(['title', 'description', 'filename', 'section'])


In [16]:
import sys
print(sys.executable)

C:\Users\user\Documents\Repo doc processor\course\.venv\Scripts\python.exe


In [17]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [18]:
from openai import OpenAI

openai_client = OpenAI()

def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text

In [19]:
prompt_template = """
Split the provided document into logical sections that make sense for a Q&A system.

Each section should be self-contained and cover a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

In [20]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [19]:
# DO NOT RUN

from tqdm.auto import tqdm

evidently_chunks = []
for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

  0%|          | 0/95 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [19]:
prompt2 = """Summarise the code in plain English. Briefly describe each class and function/method (their purpose and role) \
then give a short overall summary of how they work together. Avoid low-level details."""

In [38]:
print(prompt2)


Summarise the code in plain English. Briefly describe each class and function/method (their purpose and role) then give a short overall summary of how they work together. Avoid low-level details.


In [21]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [22]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)

In [23]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "context"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [33]:
from sentence_transformers import SentenceTransformer
import pickle

embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [25]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

In [26]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

In [27]:
similarity = v_query.dot(v_doc)

In [28]:
print(similarity)

0.007453814


In [34]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)

np.save("faq_embeddings.npy", faq_embeddings)

  0%|          | 0/402 [00:00<?, ?it/s]

In [30]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [31]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)

In [35]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    v = embedding_model.encode(d['section'])
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)
np.save("evidently_embeddings.npy", evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/264 [00:00<?, ?it/s]

In [31]:
print(evidently_chunks[0].keys())

dict_keys(['title', 'description', 'filename', 'section'])


In [32]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results = 5)

final_results = text_results + vector_results

In [36]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
            
    return combined_results

In [50]:
print(response.output)

[ResponseOutputMessage(id='msg_09e8f43c6ac7b40e006a09f048d10c81949dde93c7cc974b29', content=[ResponseOutputText(annotations=[], text="Whether you can join a course at this time depends on a few factors such as the course start date, enrollment policies, and any prerequisites that might be required. I recommend checking the course's official website or contacting the instructor or institution offering the course for specific information on enrollment options. If you provide more details about the course, I might be able to assist you better!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]


In [35]:
import openai

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages
)

print(response.output_text)

Whether you can join a course at this time depends on a few factors such as the course start date, enrollment policies, and any prerequisites that might be required. I recommend checking the course's official website or contacting the instructor or institution offering the course for specific information on enrollment options. If you provide more details about the course, I might be able to assist you better!


In [39]:
#%pprint

from pprint import pprint

pprint(response.output_text)

NameError: name 'response' is not defined

In [40]:
%pprint

from pprint import pprint

pprint(response.output_text)


Pretty printing has been turned ON
('Whether you can join a course at this time depends on a few factors such as '
 'the course start date, enrollment policies, and any prerequisites that might '
 "be required. I recommend checking the course's official website or "
 'contacting the instructor or institution offering the course for specific '
 'information on enrollment options. If you provide more details about the '
 'course, I might be able to assist you better!')


In [41]:
print(response.usage)

ResponseUsage(input_tokens=18, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=75, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=93)


In [37]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [40]:
system_prompt = """
You are a helpful assistant for a course.
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool],
    max_output_tokens=50
)

In [53]:
pprint(response.output)

[ResponseFunctionToolCall(arguments='{"query":"join course enrollment"}', call_id='call_JiSVuv2QpTMQ9IXxBW18on8W', name='text_search', type='function_call', id='fc_0d59815a3bf48a66006a09fc8842008194bbe704b8b3994244', namespace=None, status='completed')]


In [41]:
pprint(response.usage)

ResponseUsage(input_tokens=73, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=17, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=90)


In [42]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}

In [43]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool],
    max_output_tokens=50
)

print(response.output_text)

Yes, you can still join the course even if you missed the registration deadline. You’re eligible to submit homework assignments. However, keep in mind that there are deadlines for turning in homework and final projects, so it's best not to procrastinate.




In [44]:
pprint(response.usage)

ResponseUsage(input_tokens=864, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=914)


In [56]:
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""


In [51]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str) : The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)

In [52]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)

In [48]:
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

In [49]:
print(result)

AgentRunResult(output="Yes, you can still join the course even if it has already started. Although you may not be able to register officially, you are still eligible to submit the homework. Be mindful of the deadlines for homework and final projects, so try not to leave everything until the last minute. \n\nFor more detailed information, you can check the course's official resources and announcements.")


In [50]:
print(result.usage)

RunUsage(input_tokens=945, output_tokens=89, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=2, tool_calls=1)


In [42]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 5, 17, 20, 57, 16, 116481, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 17, 20, 57, 16, 117384, tzinfo=datetime.timezone.utc), instructions='You are a helpful assistant for a course.', run_id='019e37ba-9845-765b-b186-7dbbfae9e12f', conversation_id='019e37ba-9845-765b-b186-7dba306f20cb'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"join course"}', tool_call_id='call_yDc7vUzxnLFcLk4W8VYM6Usj')], usage=RequestUsage(input_tokens=105, output_tokens=15, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 5, 17, 20, 57, 18, 129962, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://api.openai.com/v1/', provider_details={'finish_reason': 'tool_calls', 